In [2]:
import spaTrack as spt
import scanpy as sc
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.pyplot as plt
from matplotlib_scalebar.scalebar import ScaleBar
import numpy as np
import os

import scanpy as sc
import anndata as ad
from matplotlib.gridspec import GridSpec
import matplotlib.pyplot as plt
from matplotlib_scalebar.scalebar import ScaleBar
from matplotlib.colors import ListedColormap, rgb2hex
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar
import matplotlib.font_manager as fm
import numpy as np
import warnings
import pandas as pd
warnings.filterwarnings('ignore')
import numpy as np
from sklearn.metrics import jaccard_score
import seaborn as sns
import matplotlib.pyplot as plt
plt.rcParams['pdf.fonttype'] = 42 # ADOBE AI 字帖
import os
from matplotlib.font_manager import fontManager, FontProperties
import gc
from scipy.sparse import csr_matrix
fontManager.addfont('/data/work/Arial.ttf')

font = FontProperties(fname='/data/work/Arial.ttf')
font_name = font.get_name()
plt.rcParams['font.family'] = font_name

sc.settings.verbosity = 0

In [3]:
def plots(adata, slide_id, path):
    os.makedirs(f'/data/work/22.fusemap/08.spatrack/script_20251011/tracks_plot_03/{slide_id}', exist_ok = True)
    colormap = {
    'Ob-Neu-1': '#8963AD',
'Ob-Neu-2': '#8CC494',
'Ob-Neu-3': '#DAB063',
'Ob-Neu-4': '#FE972B',
'Ob-Neu-5': '#EB4647',
'Ob-Neu-6': '#704599',
'Ob-Neu-7': '#84BF96',
'Ob-Neu-8': '#F16868',
'Ob-Glia-1': '#936FB3',
'Ob-Neu-15': '#B49C72',
'Ob-Neu-9': '#E2C26F',
'Ob-Neu-10': '#9CD47A',
'Ob-Neu-11': '#845DAA',
'Ob-Glia-2': '#F8A05F',
'Ob-Neu-12': '#D5A6A5',
'Ob-Neu-13': '#B79ACA',
'Ob-Neu-14': '#1F78B3',
'Ob-Glia-3': '#8FCD70',
'Ob-Vasc-1': '#AE87FF',
'Epend-1': '#90C0DB',
'Epend-2': '#C99B7E',
'Epend-3': '#FBB268',
'Epend-4': '#6A9E4A',
'Epend-5': '#AA9C6C',
'Epend-6': '#ABDA8B',
'Epend-7': '#2C80B8',
'Epend-8': '#FE8002',
}
    linewidth = 1
    density = 3
    arrow_color = 'k'
    arrow_size = 1
    arrow_style = '-|>'
    max_length = 4
    integration_direction = "both"
    # lengths = np.sqrt((adata.uns['V_grid']**2).sum(0))
    lengths = np.sqrt((adata.uns['V_grid']**2).sum(0))
    lengths = np.nan_to_num(lengths, nan=0.0, posinf=0.0, neginf=0.0)
    linewidth *= 2 * lengths / lengths[~np.isnan(lengths)].max()
    stream_kwargs = {
    "linewidth": linewidth,
    "density": density or 2,
    "zorder": 3,
    'arrow_color': 'k',
    "arrowsize": arrow_size or 1,
    "arrowstyle": arrow_style or "-|>",
    "maxlength": max_length or 4,
    "integration_direction": integration_direction or "both",
    }
    stream_kwargs["color"] = stream_kwargs.pop("arrow_color", "k")
    fig, axs = plt.subplots(figsize=(5, 5))
    plot = sc.pl.embedding(adata, basis='spatial',show=False,title=slide_id,color='ptime',ax=axs,frameon=False,s = 30, cmap = 'Reds')
    plt.axis('off');
    plot.set_aspect('equal');
    scalebar = ScaleBar(0.25, "mm", fixed_value=0.5, location = 'lower right', frameon = False,);
    plot.add_artist(scalebar);
    axs.quiver(adata.uns['E_grid'][0],adata.uns['E_grid'][1],adata.uns['V_grid'][0],adata.uns['V_grid'][1], scale=0.08)
    plt.savefig(f'{path}{slide_id}/quiver_ptime.pdf', bbox_inches = 'tight')
    plt.close()
    fig, axs = plt.subplots(figsize=(5, 5))
    plot = sc.pl.embedding(adata, basis='spatial',show=False,title=slide_id,color='cluster',ax=axs,frameon=False,s = 30, palette = colormap)
    plt.axis('off');
    plot.set_aspect('equal');
    scalebar = ScaleBar(0.25, "mm", fixed_value=0.5, location = 'lower right', frameon = False,);
    plot.add_artist(scalebar);
    axs.quiver(adata.uns['E_grid'][0],adata.uns['E_grid'][1],adata.uns['V_grid'][0],adata.uns['V_grid'][1], scale=0.08)
    plt.savefig(f'{path}{slide_id}/quiver_celltype.pdf', bbox_inches = 'tight')
    plt.close()


    fig, axs = plt.subplots(figsize=(5, 5))
    plot = sc.pl.embedding(adata, basis='spatial',show=False,title=slide_id,color='ptime',ax=axs,frameon=False,s = 30, cmap = 'Reds')
    plt.axis('off');
    plot.set_aspect('equal');
    scalebar = ScaleBar(0.25, "mm", fixed_value=0.5, location = 'lower right', frameon = False,);
    plot.add_artist(scalebar);
    axs.streamplot(adata.uns['E_grid'][0], adata.uns['E_grid'][1], adata.uns['V_grid'][0], adata.uns['V_grid'][1],**stream_kwargs)
    plt.savefig(f'{path}{slide_id}/streamplot_ptime.pdf', bbox_inches = 'tight')
    plt.close()


    fig, axs = plt.subplots(figsize=(5, 5))
    plot = sc.pl.embedding(adata, basis='spatial',show=False,title=slide_id,color='cluster',ax=axs,frameon=False,s = 30, palette = colormap)
    plt.axis('off');
    plot.set_aspect('equal');
    scalebar = ScaleBar(0.25, "mm", fixed_value=0.5, location = 'lower right', frameon = False,);
    plot.add_artist(scalebar);
    axs.streamplot(adata.uns['E_grid'][0], adata.uns['E_grid'][1], adata.uns['V_grid'][0], adata.uns['V_grid'][1],**stream_kwargs)
    plt.savefig(f'{path}{slide_id}/streamplot_celltype.pdf', bbox_inches = 'tight')
    plt.close()
    return 

In [4]:
for slice_code in ['11', '12', '13', '27', '28', '29']:
    
    adata = sc.read_h5ad(f'/data/work/22.fusemap/08.spatrack/script_20251011/tracks_plot_03/data/{slice_code}.h5ad')
    path = '/data/work/22.fusemap/08.spatrack/script_20251011/tracks_plot_03/'
    slide_id = f'slice_{slice_code}'
    plots(adata, slide_id, path)

In [5]:
colormap = {
    'Ob-Neu-1': '#8963AD',
'Ob-Neu-2': '#8CC494',
'Ob-Neu-3': '#DAB063',
'Ob-Neu-4': '#FE972B',
'Ob-Neu-5': '#EB4647',
'Ob-Neu-6': '#704599',
'Ob-Neu-7': '#84BF96',
'Ob-Neu-8': '#F16868',
'Ob-Glia-1': '#936FB3',
'Ob-Neu-15': '#B49C72',
'Ob-Neu-9': '#E2C26F',
'Ob-Neu-10': '#9CD47A',
'Ob-Neu-11': '#845DAA',
'Ob-Glia-2': '#F8A05F',
'Ob-Neu-12': '#D5A6A5',
'Ob-Neu-13': '#B79ACA',
'Ob-Neu-14': '#1F78B3',
'Ob-Glia-3': '#8FCD70',
'Ob-Vasc-1': '#AE87FF',
'Epend-1': '#90C0DB',
'Epend-2': '#C99B7E',
'Epend-3': '#FBB268',
'Epend-4': '#6A9E4A',
'Epend-5': '#AA9C6C',
'Epend-6': '#ABDA8B',
'Epend-7': '#2C80B8',
'Epend-8': '#FE8002',
}
for slice_code in ['11', '12', '13', '27', '28', '29']:
    adata = sc.read_h5ad(f'/data/work/22.fusemap/08.spatrack/script_20251011/tracks_plot_03/data/{slice_code}.h5ad')
    adata.obs.to_csv(f'/data/work/22.fusemap/08.spatrack/script_20251011/tracks_plot_03/data/{slice_code}.csv')
    adata.obs['celltype'] = adata.obs['cluster'].tolist()
    adata.obs['celltype'] = adata.obs['celltype'].astype('category')
    celltypes = adata.obs['celltype'].cat.categories
    adata.uns['celltype_colors'] = [colormap[ct] for ct in celltypes]
    ax = sc.pl.violin(adata, keys= 'ptime', groupby = 'celltype', rotation = 45, show = False)
    for tick in ax.get_xticklabels():
        tick.set_ha('right')
    slide_id = f'slice_{slice_code}'
    plt.savefig(f'/data/work/22.fusemap/08.spatrack/script_20251011/tracks_plot_03/{slide_id}/violinplot_ptime.pdf', bbox_inches= 'tight')
    plt.close()
    
    
    ax = sc.pl.violin(adata, keys= 'entropy', groupby = 'celltype', rotation = 45, show = False)
    for tick in ax.get_xticklabels():
        tick.set_ha('right')
    slide_id = f'slice_{slice_code}'
    plt.savefig(f'/data/work/22.fusemap/08.spatrack/script_20251011/tracks_plot_03/{slide_id}/violinplot_entropy.pdf', bbox_inches= 'tight')
    plt.close()

In [6]:
adata

AnnData object with n_obs × n_vars = 2014 × 12212
    obs: 'region', 'slices', 'z', 'cluster', 'level1', 'level0', 'level-1', 'new_cluster', 'anno_whole_cluster', 'ax', 'ay', 'az', 'entropy', 'tran', 'ptime', 'celltype'
    uns: 'E_grid', 'P_grid', 'V_grid', 'entropy value', 'entropy value order', 'pca', 'celltype_colors'
    obsm: '3d_align_spatial_confine', 'X_pca', 'X_spatial', 'spatial', 'velocity_spatial'
    varm: 'PCs'
    obsp: 'trans', 'trans_neigh_csr'